This notebook is used to test out ways to extract the AS train file blobs.

In [2]:
import os
import zipfile as zf
import numpy as np
import argparse
import h5py
import librosa as lb
import torchaudio
import utils.config as config
import utils.utility_funcs as my_utils

In [ ]:
# Misbehaves in notebooks, works in .py files
# Initialize an argument parser to relay filenames on the command line
"""
parser = argparse.ArgumentParser(description="A script to extract Audioset files " \
"from a directory containing several blob files and extracting to destination directory based on an " \
"external text file of valid filenames. Archive's file structure is ignored in the process.")
parser.add_argument("--blob_dir", help="Path of the blob directory")
parser.add_argument("--valid_filenames", help="Text file containing valid filenames separated by '\n' ")
parser.add_argument("--target_dir", help="The destination of the extracted files")

args = parser.parse_args()
"""

In [3]:
# Read in the desired filenames
PATH_TO_AS_TRAIN_FNAMES = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\audioset_train_filenames_top50.txt'
valid_filenames = set()
with open(PATH_TO_AS_TRAIN_FNAMES, 'r') as fnames:
    for line in fnames.readlines():
        valid_filenames.add(line.rstrip('\n'))

In [7]:
# These same files are found on a computing cluster, testing locally first 
PATH_TO_LOCAL_AUDIOSET_00_ZIP = r'C:\Users\mp431591\Documents\audioset_unbalanced_test\unbalanced_train_segments_part00_full.zip'
PATH_TO_TEST_DEST = r'C:\Users\mp431591\Documents\audioset_unbalanced_test\test_dest'
PATH_TO_BLOB_DIR = r'C:\Users\mp431591\Documents\audioset_unbalanced_test\faux_blob_folder_for_testing'

MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(sample_rate=config.sample_rate,
                                                     n_fft=config.n_fft,
                                                     hop_length=config.hop_length,
                                                     n_mels=config.n_mels,
                                                     f_min=config.fmin,
                                                     f_max=config.fmax)

with os.scandir(PATH_TO_BLOB_DIR) as blob_dir:
    for entry in blob_dir:

        with zf.ZipFile(entry.path, 'r') as archive:
            archive_path = archive.filename

            for zip_info in archive.infolist()[0:18]:

                if zip_info.is_dir():
                    continue
                
                zip_info.filename = os.path.basename(zip_info.filename) # Break directories
                filename = zip_info.filename
                clean_file_name = filename.rstrip(".wav")

                if clean_file_name in valid_filenames:
                    # If extracting
                    #archive.extract(zip_info, path=PATH_TO_TEST_DEST)

                    # If processing
                    with archive.open(zip_info, 'r') as audio_file:
                        target_sr = config.sample_rate
                        audio, sr = torchaudio.load(audio_file)
                        print(audio.shape)
                        if sr != target_sr:
                            print(f"Resampling {clean_file_name}")
                            audio = torchaudio.functional.resample(audio, 
                                                           orig_freq=sr,
                                                           new_freq=target_sr)
                        audio = my_utils.pad_or_truncate(audio, target_sr * 10)
                        mel = MEL_TRANSFORM(audio)
                        print(mel.shape)




torch.Size([1, 320000])
torch.Size([1, 64, 1001])
torch.Size([1, 320000])
torch.Size([1, 64, 1001])
